# Real LoRA Training in C++ Using LibTorch

This notebook demonstrates a realistic LoRA implementation using PyTorch C++ (LibTorch).

Why LibTorch on RK3588?

- Mature ecosystem
- Same APIs and concepts as PyTorch
- Easier to understand PEFT internals
- Can train on PC and deploy to RKLLM later

This notebook focuses on the core LoRA math, not transformers.

## LoRA Formula

Base layer:

`y = Wx`

LoRA layer:

`y = (W + BA)x`

Only A and B are trainable.

In [1]:
#include <torch/torch.h>
#include <iostream>

std::cout << TORCH_VERSION << std::endl;

In file included from <<< inputs >>>:1:
input_line_1:1:10: fatal error: 'torch/torch.h' file not found
    1 | #include <torch/torch.h>
      |          ^~~~~~~~~~~~~~~
Failed to parse via ::process:Parsing failed.


Error: : Compilation error! In file included from <<< inputs >>>:1:
[1minput_line_1:1:10: [0m[0;1;31mfatal error: [0m[1m'torch/torch.h' file not found[0m
    1 | #include <torch/torch.h>[0m
      | [0;1;32m         ^~~~~~~~~~~~~~~
[0mFailed to parse via ::process:Parsing failed.


In [ ]:
struct LoRALinearImpl : torch::nn::Module
{
    torch::Tensor W;

    torch::Tensor A;
    torch::Tensor B;

    int rank;

    LoRALinearImpl(
        int in_features,
        int out_features,
        int r=2)
        : rank(r)
    {
        W = register_buffer(
            "W",
            torch::randn({out_features,in_features})
        );

        A = register_parameter(
            "A",
            torch::randn({rank,in_features}) * 0.01
        );

        B = register_parameter(
            "B",
            torch::zeros({out_features,rank})
        );
    }

    torch::Tensor forward(torch::Tensor x)
    {
        auto delta =
            torch::matmul(B,A);

        auto merged =
            W + delta;

        return torch::matmul(
            x,
            merged.t()
        );
    }
};

TORCH_MODULE(LoRALinear);

## Create Dataset

In [ ]:
auto x =
torch::tensor({
    {1.0},
    {2.0},
    {3.0},
    {4.0}
});

auto y =
torch::tensor({
    {2.0},
    {4.0},
    {6.0},
    {8.0}
});

## Train Only LoRA Parameters

In [ ]:
LoRALinear model(1,1,1);

model->W.fill_(1.0);

torch::optim::Adam optimizer(
    model->parameters(),
    torch::optim::AdamOptions(0.01)
);

for(int epoch=0; epoch<1000; ++epoch)
{
    optimizer.zero_grad();

    auto pred =
        model->forward(x);

    auto loss =
        torch::mse_loss(
            pred,
            y
        );

    loss.backward();

    optimizer.step();

    if(epoch % 100 == 0)
    {
        std::cout
            << "epoch="
            << epoch
            << " loss="
            << loss.item<float>()
            << std::endl;
    }
}

## Inspect Learned Adapter

In [ ]:
std::cout << "A\n"
          << model->A
          << std::endl;

std::cout << "B\n"
          << model->B
          << std::endl;

auto delta =
torch::matmul(
    model->B,
    model->A
);

std::cout
    << "Delta W\n"
    << delta
    << std::endl;

## Test

In [ ]:
auto pred =
model->forward(x);

std::cout
    << pred
    << std::endl;

## Relation To Real LLM LoRA

Real PEFT LoRA does the same thing.

Instead of a tiny linear layer:

- q_proj
- k_proj
- v_proj
- o_proj

receive trainable A/B matrices.

Typical workflow:

1. Train LoRA with PyTorch.
2. Save adapter.
3. Merge adapter.
4. Export Hugging Face model.
5. Convert to RKLLM.
6. Deploy on RK3588.